# Taking Input
Giving the user two choices of input.
They can either paste the URL of a website containing the article like Wikipedia or they can input the text to be summarised through the input field.

In [42]:
print("Please choose your prefered way to input text.\nPress 1 to copy paste the URL of a website.\nPress 2 to input your own text.\n")
input_choice = input()
summary_size = (input("Enter the lines you want in the summary.\n"))
if input_choice == "1":
    url = input("\nPlease enter the URL:\n")
    #importing the necessary libraries to parse the data from the url
    import bs4 as bs
    import urllib.request
    import re
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.0 Safari/605.1.15'}
    req = urllib.request.Request(url, headers=headers)
    scraped_data = urllib.request.urlopen(req) 
    #reading the data from the object returned by the urlopen
    article = scraped_data.read()
    #parsing the article using BeautifulSoup
    parsed_article = bs.BeautifulSoup(article,'lxml')
    #using findall function on the object returned by BeautifulSoup to get the paragraphs
    paragraphs = parsed_article.find_all('p')
    article_text = ""
    for p in paragraphs:
        article_text += p.text
else:
 article_text = input("Please enter your desired text.\n")

Please choose your prefered way to input text.
Press 1 to copy paste the URL of a website.
Press 2 to input your own text.



Ensure article_text is not empty


In [43]:
if not article_text.strip():
    raise ValueError("The input text is empty. Please provide valid text.")

In [44]:
#importing PythonRegex(Regular Expression)
import re
# Removing Square Brackets and Extra Spaces and replacing them with white spaces
article_text = re.sub(r'\[[0-9]*\]', ' ', article_text)  
article_text = re.sub(r'\s+', ' ', article_text) 
article_text = re.sub(r'[^\x00-\x7F]+', ' ', article_text)  # Remove non-ASCII characters
article_text = re.sub(r'\s+', ' ', article_text)  # Remove extra spaces

In [45]:
# Removing special characters and digits
formatted_article_text = re.sub('[^a-zA-Z]', ' ', article_text )  
formatted_article_text = re.sub(r'\s+', ' ', formatted_article_text) 
print("Formatted Article Text:", formatted_article_text)


Formatted Article Text:  Bad Times at the El Royale is a American neo noir hyperlink thriller film written directed and produced by Drew Goddard Starring Jeff Bridges Cynthia Erivo Dakota Johnson Jon Hamm Cailee Spaeny Lewis Pullman and Chris Hemsworth the film follows six strangers and an employee at the El Royale a hotel located along the California Nevada border whose secrets intersect on a night in the late s The film explores themes of morality faith and redemption with the state border and other visual elements symbolizing the concept of right and wrong Goddard began writing the spec script for the film in November and compiled a list of songs into his screenplay After telling major studios to avoid buying the script if they could not buy the licenses for each piece of music he sold it to th Century Fox in March Principal photography began on January with cinematographer Seamus McGarvey and concluded on April The El Royale hotel was built entirely on a studio set in Burnaby under

# Conveting the processed text into sentences 

In [46]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
try:
    sentence_list = nltk.sent_tokenize(article_text)
    print("Sentence List:", sentence_list)
except Exception as e:
    print("Error in nltk.sent_tokenize:", str(e))
#sentence_list = nltk.sent_tokenize(article_text) 

Sentence List: [' Bad Times at the El Royale is a 2018 American neo-noir hyperlink thriller film written, directed, and produced by Drew Goddard.', 'Starring Jeff Bridges, Cynthia Erivo, Dakota Johnson, Jon Hamm, Cailee Spaeny, Lewis Pullman, and Chris Hemsworth, the film follows six strangers and an employee at the El Royale, a hotel located along the California Nevada border, whose secrets intersect on a night in the late 1960s.', 'The film explores themes of morality, faith, and redemption, with the state border and other visual elements symbolizing the concept of right and wrong.', 'Goddard began writing the spec script for the film in November 2016, and compiled a list of songs into his screenplay.', 'After telling major studios to avoid buying the script if they could not buy the licenses for each piece of music, he sold it to 20th Century Fox in March 2017.', 'Principal photography began on January 29, 2018, with cinematographer Seamus McGarvey, and concluded on April 6.', 'The 

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/kashyaphebbar/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kashyaphebbar/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Finding the frequency of occurency of words along with their weights

In [47]:
import nltk
nltk.download('stopwords')


stopwords = nltk.corpus.stopwords.words('english')

word_frequencies = {}  
for word in nltk.word_tokenize(formatted_article_text):  
    if word not in stopwords:
        if word not in word_frequencies.keys():
            word_frequencies[word] = 1
        else:
            word_frequencies[word] += 1

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kashyaphebbar/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [48]:
maximum_frequncy = max(word_frequencies.values())

for word in word_frequencies.keys():  
    word_frequencies[word] = (word_frequencies[word]/maximum_frequncy)

# Calculating the scores of the sentences

In [49]:
sentence_scores = {}  
for sent in sentence_list:  
    for word in nltk.word_tokenize(sent.lower()):
        if word in word_frequencies.keys():
            if len(sent.split(' ')) < 30:
                if sent not in sentence_scores.keys():
                    sentence_scores[sent] = word_frequencies[word]
                else:
                    sentence_scores[sent] += word_frequencies[word]

# Getting the Summary

In [50]:
import heapq

# Ensure summary_size is an integer
summary_size = int(summary_size)

# Debug: Check sentence_scores
print("Sentence Scores:", sentence_scores)

# Ensure sentence_scores is not empty
if not sentence_scores:
    raise ValueError("No sentence scores were calculated. Ensure the input text is valid.")

# Limit summary_size to the number of sentences
summary_size = min(summary_size, len(sentence_scores))

# Get the summary
summary_sentences = heapq.nlargest(summary_size, sentence_scores, key=sentence_scores.get)
summary = ' '.join(summary_sentences)
print("Summary:", summary)

Sentence Scores: {' Bad Times at the El Royale is a 2018 American neo-noir hyperlink thriller film written, directed, and produced by Drew Goddard.': 1.1573033707865168, 'The film explores themes of morality, faith, and redemption, with the state border and other visual elements symbolizing the concept of right and wrong.': 1.3483146067415737, 'Goddard began writing the spec script for the film in November 2016, and compiled a list of songs into his screenplay.': 1.449438202247191, 'Principal photography began on January 29, 2018, with cinematographer Seamus McGarvey, and concluded on April 6.': 0.16853932584269662, 'The El Royale hotel was built entirely on a studio set in Burnaby, under the supervision of production designer Martin Whist, who had envisioned designing a perfectly symmetrical hotel.': 0.8651685393258428, 'During post-production, editing was completed by Lisa Lassek and the musical score was composed by Michael Giacchino.': 0.11235955056179775, "Bad Times at the El Roya

In [52]:
import nltk

# Save scraped article line by line
with open("scraped_article.txt", "w", encoding="utf-8") as f:
    for sentence in nltk.sent_tokenize(article_text):
        f.write(sentence.strip() + "\n")

# Save summary line by line
with open("summary.txt", "w", encoding="utf-8") as f:
    for sentence in nltk.sent_tokenize(summary):
        f.write(sentence.strip() + "\n")

print("Scraped article and summary have been saved line by line as 'scraped_article.txt' and 'summary.txt'.")

Scraped article and summary have been saved line by line as 'scraped_article.txt' and 'summary.txt'.
